# Notebook 01 — Data Understanding

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction using GraphSAGE

### Objective
This notebook performs **data understanding only** on the official UCI Bank Marketing `bank-full.csv` dataset.

We will:
- Load the raw dataset without modifying it.
- Inspect its shape and schema.
- Identify numerical and categorical features.
- Identify the target variable.
- Check missing values and duplicates.
- Inspect unique values and basic data quality.
- Analyze the target distribution.
- Produce concise, useful summaries for the next project stage.

### Important boundary
This notebook does **not** perform:
- model training
- graph construction
- feature scaling
- categorical encoding
- train/validation/test splitting
- heavy feature engineering

Those tasks belong to later notebooks.


## Project Input

Expected raw dataset path:

```text
Bank-Marketing-GNN/
└── data/
    └── raw/
        └── bank-full.csv
```

The notebook uses a project-root-relative path so it can be run from Jupyter Notebook, VS Code, or JupyterLab with minimal path changes.


In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

RANDOM_STATE = 42

# Locate the project root relative to this notebook.
PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is executed from the project root, this is correct.
# If executed from the notebooks directory, move one level up.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"
ARTIFACTS_METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)


In [ ]:
# ============================================================
# 2. Verify Dataset File
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Please place the already-downloaded bank-full.csv file in "
        "data/raw/ before running this notebook."
    )

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset path is not a file: {DATA_PATH}")

file_size_mb = DATA_PATH.stat().st_size / (1024 ** 2)

print("Dataset file verified successfully.")
print(f"File: {DATA_PATH.name}")
print(f"Size: {file_size_mb:.2f} MB")


## 3. Load the Raw Dataset

The UCI Bank Marketing dataset uses a semicolon (`;`) delimiter.

We explicitly specify the delimiter rather than relying on automatic inference so that the schema is validated rather than assumed.


In [ ]:
# ============================================================
# 3. Load Dataset
# ============================================================

df = pd.read_csv(DATA_PATH, sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


In [ ]:
# ============================================================
# 4. Basic Dataset Dimensions
# ============================================================

n_rows, n_columns = df.shape

print(f"Number of rows    : {n_rows:,}")
print(f"Number of columns : {n_columns:,}")


## 5. Column and Schema Inspection

The purpose of this section is to establish the raw schema that later preprocessing must preserve and handle consistently.


In [ ]:
# ============================================================
# 5. Schema Inspection
# ============================================================

schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "unique_count": df.nunique(dropna=False).values,
})

schema_df["null_percentage"] = (
    schema_df["null_count"] / len(df) * 100
).round(2)

display(schema_df)


In [ ]:
# ============================================================
# 6. Data Types
# ============================================================

print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nNumerical columns:")
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
print(numerical_columns)

print("\nCategorical/object columns:")
categorical_columns = df.select_dtypes(include=["object", "category"]).columns.tolist()
print(categorical_columns)


## 7. Identify the Target

The project target is expected to be:

```text
y
```

where the raw values represent whether the customer subscribed to a term deposit.


In [ ]:
# ============================================================
# 7. Target Identification
# ============================================================

TARGET_COLUMN = "y"

if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Expected target column '{TARGET_COLUMN}' was not found. "
        f"Available columns: {df.columns.tolist()}"
    )

print(f"Target column: {TARGET_COLUMN}")
print("Target dtype :", df[TARGET_COLUMN].dtype)
print("Target values:", df[TARGET_COLUMN].dropna().unique().tolist())


In [ ]:
# ============================================================
# 8. Target Distribution
# ============================================================

target_counts = df[TARGET_COLUMN].value_counts(dropna=False)
target_percentages = (
    df[TARGET_COLUMN]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

display(target_distribution)

print("Target distribution plot:")

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x=TARGET_COLUMN)
plt.title("Target Distribution")
plt.xlabel("Subscription Outcome")
plt.ylabel("Customer Count")
plt.tight_layout()
plt.show()


## 9. Missing-Value Analysis

This checks the raw dataset for explicit missing values.

A zero missing-value count does not mean every value is semantically valid; later notebooks will perform more detailed preprocessing checks.


In [ ]:
# ============================================================
# 9. Missing Values
# ============================================================

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_count", ascending=False
)

display(missing_summary)

total_missing = int(df.isna().sum().sum())
print(f"Total missing values: {total_missing:,}")


## 10. Duplicate Analysis

Duplicates are reported here but the raw dataset is **not modified** in this notebook.


In [ ]:
# ============================================================
# 10. Duplicate Rows
# ============================================================

duplicate_count = int(df.duplicated().sum())
duplicate_percentage = duplicate_count / len(df) * 100

print(f"Duplicate rows      : {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")


## 11. Numerical Feature Summary

This section provides descriptive statistics for numerical variables without applying any transformation.


In [ ]:
# ============================================================
# 11. Numerical Summary
# ============================================================

numerical_summary = df[numerical_columns].describe().T
display(numerical_summary)


## 12. Categorical Feature Summary

For categorical variables, inspect the number of unique values and the most frequent categories.


In [ ]:
# ============================================================
# 12. Categorical Summary
# ============================================================

categorical_summary_rows = []

for column in categorical_columns:
    value_counts = df[column].value_counts(dropna=False)

    top_value = value_counts.index[0] if len(value_counts) else None
    top_count = int(value_counts.iloc[0]) if len(value_counts) else 0

    categorical_summary_rows.append({
        "column": column,
        "unique_values": int(df[column].nunique(dropna=False)),
        "top_value": top_value,
        "top_count": top_count,
        "top_percentage": round(top_count / len(df) * 100, 2)
    })

categorical_summary = pd.DataFrame(categorical_summary_rows)
display(categorical_summary)


## 13. Unique Values

For this dataset, categorical cardinality is important because these variables may later become node types or relations in the heterogeneous graph.

We inspect every categorical column rather than modifying it.


In [ ]:
# ============================================================
# 13. Unique Values for Categorical Features
# ============================================================

for column in categorical_columns:
    values = df[column].dropna().unique().tolist()

    print(f"\n{column} ({len(values)} unique values)")
    print(values)


## 14. Numerical Feature Distributions

These plots provide a first understanding of the raw numerical variables.

No scaling or transformation is performed here.


In [ ]:
# ============================================================
# 14. Numerical Distributions
# ============================================================

if numerical_columns:
    n_cols = 3
    n_rows = int(np.ceil(len(numerical_columns) / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(16, 4 * n_rows)
    )

    axes = np.array(axes).reshape(-1)

    for ax, column in zip(axes, numerical_columns):
        sns.histplot(df[column], kde=True, ax=ax)
        ax.set_title(f"Distribution of {column}")
        ax.set_xlabel(column)
        ax.set_ylabel("Count")

    for ax in axes[len(numerical_columns):]:
        ax.remove()

    plt.tight_layout()
    plt.show()


## 15. Basic Data Quality Checks

The following assertions provide a reproducible schema/data-quality gate for the next notebook.


In [ ]:
# ============================================================
# 15. Data Quality Checks
# ============================================================

assert len(df) > 0, "Dataset contains no rows."
assert len(df.columns) > 0, "Dataset contains no columns."
assert TARGET_COLUMN in df.columns, "Target column is missing."
assert df.columns.is_unique, "Column names are not unique."

print("Basic data quality assertions passed.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Target: {TARGET_COLUMN}")


## 16. Feature Groups for the Next Stages

This is an **inspection-only classification** based on the raw pandas dtypes.

The final preprocessing and graph-construction notebooks will explicitly define and validate their own feature schemas.


In [ ]:
# ============================================================
# 16. Feature Groups
# ============================================================

feature_columns = [column for column in df.columns if column != TARGET_COLUMN]

numerical_features = [
    column for column in feature_columns
    if pd.api.types.is_numeric_dtype(df[column])
]

categorical_features = [
    column for column in feature_columns
    if not pd.api.types.is_numeric_dtype(df[column])
]

feature_group_summary = {
    "target_column": TARGET_COLUMN,
    "total_features": len(feature_columns),
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
}

print("Feature group summary:")
print(json.dumps(feature_group_summary, indent=2))


## 17. Save Raw Schema Metadata

Only lightweight metadata is saved at this stage.

The raw CSV itself is **not modified**.

This metadata is useful for validating future preprocessing steps.


In [ ]:
# ============================================================
# 17. Save Metadata
# ============================================================

ARTIFACTS_METADATA_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "dataset": {
        "filename": DATA_PATH.name,
        "delimiter": ";",
        "rows": int(n_rows),
        "columns": int(n_columns),
    },
    "target": {
        "column": TARGET_COLUMN,
        "dtype": str(df[TARGET_COLUMN].dtype),
        "unique_values": [
            None if pd.isna(value) else str(value)
            for value in df[TARGET_COLUMN].dropna().unique()
        ],
        "distribution": {
            str(key): int(value)
            for key, value in target_counts.items()
        }
    },
    "features": {
        "all_features": feature_columns,
        "numerical_features": numerical_features,
        "categorical_features": categorical_features,
    },
    "quality": {
        "total_missing_values": total_missing,
        "duplicate_rows": duplicate_count,
    }
}

metadata_path = ARTIFACTS_METADATA_DIR / "data_understanding_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")


# Final Verification

Notebook 01 is considered successful only when all of the following checks pass:

- [x] Raw dataset loads successfully.
- [x] Dataset shape is available.
- [x] Column schema is available.
- [x] Numerical and categorical features are identified.
- [x] Target column `y` is identified.
- [x] Target distribution is displayed.
- [x] Missing-value summary is generated.
- [x] Duplicate-row count is generated.
- [x] Numerical summary is generated.
- [x] Categorical cardinalities are inspected.
- [x] Basic data-quality assertions pass.
- [x] Data-understanding metadata is saved.

## Expected Verification

At the bottom of the executed notebook, confirm that:
1. No exception occurred.
2. The target column is `y`.
3. The printed row/column counts match the verified dataset.
4. Numerical and categorical feature lists are sensible.
5. `artifacts/metadata/data_understanding_metadata.json` exists.

**Do not proceed to Notebook 02 until these outputs have been verified.**


In [ ]:
# ============================================================
# 18. Final Automated Verification
# ============================================================

assert DATA_PATH.exists(), "Raw dataset file is missing."
assert len(df) > 0, "Dataset is empty."
assert TARGET_COLUMN in df.columns, "Target column is missing."
assert metadata_path.exists(), "Metadata artifact was not created."

print("=" * 70)
print("NOTEBOOK 01 VERIFICATION PASSED")
print("=" * 70)
print(f"Dataset shape       : {df.shape}")
print(f"Target column       : {TARGET_COLUMN}")
print(f"Numerical features  : {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Missing values      : {total_missing:,}")
print(f"Duplicate rows      : {duplicate_count:,}")
print(f"Metadata artifact   : {metadata_path}")
print("=" * 70)
print("Notebook 01 complete. Stop here and verify the outputs before NEXT.")
